# Supernovae in POSYDON

<span style="font-size:15px">

<div class='alert alert-info'>

In this tutorial, you will use an already completed POSYDON population-synthesis run to explore the final outcomes of binary systems. You will learn how to:

1. **Select systems of interest**
   - Filter the population to identify binaries that reach a final evolutionary endpoint.
   - Extract relevant information for systems that produce different compact-object and supernova outcomes.

    <br>     
2. **Determine collapse mechanisms**
   - Identify different collapse outcomes (e.g., CCSNe, ECSNe, PISN)

     <br>
3. **Connect simulations to observable outcomes**
   - Determine the predicted observed supernova types they produce.
   - Relate POSYDON model outputs to observed supernova populations.      

<br>
 
4. **Calculate ejecta masses** 
   
5. **Explore evolutionary channels**
    - Identify their progenitor evolutionary pathways and mass transfer history.
  
</div>
</span>

#### Managing Large Binary Populations and Identifying Stellar End States

<span style="font-size:15px">
In your own research with POSYDON, you will likely work with much larger populations than the small number of systems used in this tutorial. Large populations are essential for obtaining statistically meaningful results and studying the properties of binary populations. In this session, you will learn how to work with such datasets. We will begin by exploring the columns stored in the POSYDON population output (saved in HDF5, <code>.h5</code>, files) and then examine the evolutionary history of a representative binary system.
</span>

#### Loading Your Population

In [ ]:
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import os
import posydon
import pandas as pd

shared_file = (
    Path.home()
    / "data"
    / "Populations"
    / "5K_pops"
    / "SNe_lab1.h5"
)

local_file = str(Path.cwd() / shared_file.name)

shutil.copy2(shared_file, local_file)

In [ ]:
from posydon.popsyn.synthetic_population import Population
pop = Population(local_file, chunksize=1000) 

For more info: https://posydon.org/POSYDON/latest/api_reference/posydon.popsyn.html#module-posydon.popsyn.synthetic_population

In [ ]:
# the parameter space and initial conditions of the population file
pop.ini_params

<span style="font-size:15px">

Each POSYDON population file contains three main datasets: **history**, **oneline**, and **formation_channels**. In this tutorial, we will explore each of these datasets to understand the information they contain and the supernova-related properties that can be extracted from them. We begin with the **history** dataset.

</span>

In [ ]:
## Binary Properties Stored in the history dataframe of the population file 
print(pop.history.columns)


In [ ]:
#reads the history table from the HDF5 (.h5) population file and returns it as a pandas DataFrame.
df = pop.history.select()

In [ ]:
# Examining the evolution of a specific system from your population, focusing on the columns of interest.
col = ['step_names','time','state','event','S1_state','S2_state','S1_mass','S2_mass','orbital_period','separation','S1_surface_h1','S1_surface_n14','S2_surface_h1','S2_surface_n14']
df.loc[9][col]

<span style="font-size:15px">

Above, you can see the evolution of a specific binary system. You will focus on the final stages of stellar evolution by identifying the rows and columns that provide information about a star’s end of life.

In POSYDON, the end stages are marked in the `event` column with either `CC1` or `CC2`, indicating whether the collapse corresponds to the primary or secondary star. Immediately after an `event = CC1`, you will notice that the `step_names` column transitions to `step_SN`. Our first task is to establish how to systematically identify all CC1 and CC2 events across the population.

POSYDON distinguishes between two types of core-collapse events: `CC1` for primaries (star_1, the initially more massive star) and `CC2` for secondaries (star_2). For single stars, only `CC1` events occur, since they are treated as primaries. With this in mind, let’s find all the `CC1` and `CC2` events that arise from the different evolutionary channels.



<div class='alert alert-warning'>

 **Disclaimer:** For historical reasons, POSYDON uses the event labels `CC1` and `CC2` to denote the **final evolutionary stage** of the primary and secondary star, respectively. These labels do **not** necessarily indicate a core-collapse supernova. Instead, they mark the end of the stellar evolution calculated by POSYDON (typically up to carbon-core depletion for massive stars). Consequently, systems ending as **neutron stars (NSs)**, **black holes (BHs)**, **white dwarfs (WDs)**, or even **massless remnants** (e.g., following a pair-instability supernova, PISN) all pass through a `CC1` or `CC2` event.

 </div>

 </span>



<span style="font-size:15px">


Let's create a dataframe containing only the entries where the `step_names` column is equal to `'step_SN'`. By applying this mask, we can extract the properties of each supernova event, including the remnant mass, the compact object type (black hole, neutron star, white dwarf, or a massless remnant in the case of a pair-instability supernova, PISN), the post-core-collapse binary state, and the time at which the supernova occurred since the formation of the binary, assuming a starburst population.
 </span>


In [ ]:
# Identify the post-supernova (step_SN) rows corresponding to the core collapse
# of the primary star (CC1) and the secondary star (CC2).
# The actual supernova properties are stored in the `step_SN` row, while the
# previous row (`event.shift(1)`) tells us which star underwent core collapse.
post_CC1 = ((df['step_names'] == "step_SN") & (df['event'].shift(1) == "CC1"))
post_CC2 = ((df['step_names'] == "step_SN") & (df['event'].shift(1) == "CC2"))

# Extract the post-supernova properties for systems where the primary star
# (star 1) collapsed.
# We keep:
#   - time: time since binary formation
#   - state: binary state immediately after the supernova
#   - S1_state: compact object type (WD, NS, BH, etc.)
#   - S1_mass: compact remnant mass

df1 = df[["time", "state", "S1_state", "S1_mass"]][post_CC1]

# Rename columns to make their meaning explicit.
df1.rename(columns={
    'state': 'binary_state_postCC',
    'S1_state': 'stellar_state_postCC',
    'S1_mass': 'compact_object_mass'
}, inplace=True)

# Record which star produced the compact remnant.
df1["progenitor_star"] = 1

# Repeat the same procedure for systems where the secondary star (star 2)
# experienced core collapse.


df2 = df[["time", "state", "S2_state", "S2_mass"]][post_CC2]

df2.rename(columns={
    'state': 'binary_state_postCC',
    'S2_state': 'stellar_state_postCC',
    'S2_mass': 'compact_object_mass'
}, inplace=True)

# Record that the compact remnant originated from the secondary star.
df2["progenitor_star"] = 2

# Combine the core-collapse events from both stars into a single dataframe.
# Each row now corresponds to one compact-object formation event in the
# synthetic population.

df_synthetic = pd.concat([df1, df2], axis=0)

<span style="font-size:15px">


Let’s display the outcome of the SN population we just created! The table below summarizes the state of the binary after the collapse of the primary and secondary stars, along with the explosion times of both events. It also shows whether the system was disrupted or remained bound following each core-collapse, and whether the resulting remnant originated from a merger product or from a single star. In addition, it identifies the type of compact object formed and its mass (WD, NS, BH, or a massless remnant in the case of a PISN at low metallicity). In the progenitor column, a value of 1 refers to the primary star, while 2 refers to the secondary (if present).

 </span>


In [ ]:
#col_CC = ["time", "binary_state_postCC", "stellar_state_postCC", "compact_object_mass", "progenitor_star"]
df_synthetic.head(10)

<span style="font-size:15px">


Now that you are familiar with extracting information from supernova events, the following code identifies all neutron star (NS) and black hole (BH) remnants formed by the secondary stars in the binary population. Examine the code carefully to understand how these objects are selected from the df_synthetic dataframe.

</span> 


In [ ]:
# filter only NS and BH from progenitor star 2
mask = (df_synthetic["stellar_state_postCC"].isin(["NS", "BH"])) & (df_synthetic["progenitor_star"] == 2)
ns_bh_from_star2 = df_synthetic[mask]

# count NS and BH separately
count_ns = (ns_bh_from_star2["stellar_state_postCC"] == "NS").sum()
count_bh = (ns_bh_from_star2["stellar_state_postCC"] == "BH").sum()

print("From progenitor star 2:")
print("NS:", count_ns)
print("BH:", count_bh)


<div class="alert alert-success">
<span style="font-size:15px">

## 🛠️ Hands-on Exercise: Mass Distribution of Black Holes

Now it is your turn! Use what you have learned so far to explore the black hole population in the synthetic dataset.

Your tasks are:

1. **Identify black holes formed from single stars**
   - Complete the code below by replacing the **XXX** values.
   - Use the df_synthetic dataframe to select all BH objects that originated from single-star systems.

2. **Plot the black hole mass distribution**
   - Create a plot showing the distribution of the BH masses you identified.
   - Explore the properties of the resulting population.
</div>
</span>


In [ ]:

mask_singles_BH = (XXX) & (df_synthetic["XXX"].isin(["BH"]))

bh_from_singles = df_synthetic[mask_singles_BH]

bh_masses = bh_from_singles["compact_object_mass"]
plt.figure()
plt.hist(bh_masses, bins=10)
plt.xlabel(r"BH mass [$M_\odot$]")
plt.ylabel("Count")
plt.title("BH mass distribution (initially single stars)")
plt.show()


<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<details>

<b><summary>Solution </summary></b>
```
#STEP 1

mask_singles_BH = (df_synthetic["binary_state_postCC"].isin(["initially_single_star"])) & (df_synthetic["stellar_state_postCC"].isin(["BH"]))

bh_from_singles = df_synthetic[mask_singles_BH]

#STEP 2

bh_masses = bh_from_singles["compact_object_mass"]
plt.figure()
plt.hist(bh_masses, bins=10)
plt.xlabel(r"BH mass [$M_\odot$]")
plt.ylabel("Count")
plt.title("BH mass distribution (initially single stars)")
plt.show()


```
    
</details>

## Exploring the oneline DataFrame of a Population File

<span style="font-size:15px">

Some supernova-related properties, such as the explosion mechanism (e.g., ECSN or CCSN) and the total hydrogen and helium ejecta masses, are not stored in the `history` DataFrame but in the `oneline` DataFrame. These quantities are essential for classifying supernova types, as we will see later.

In this section, we will explore the information available in the `oneline` DataFrame and learn how to combine it with the `history` DataFrame to obtain a more complete picture of the supernova population.

The `oneline` DataFrame stores summary quantities for each binary system, including parameters such as `SN_type`, `h1_mass_ej`, and `he4_mass_ej`. The quantities `h1_mass_ej` and `he4_mass_ej` correspond to the total masses of hydrogen and helium contained in the supernova ejecta, respectively.

</span>


#### Printing out the column names stored in the oneline dataframe of the population

In [ ]:
pop.oneline.columns

<span style="font-size:15px">


 The oneline DataFrame stores a single summary row for each binary in the population exctracted by a POSYDON run. It retains the initial (*_i) and final (*_f) properties of the binary system and its two stellar components, together with quantities evaluated at key evolutionary stages (e.g., core helium depletion or core carbon depletion) that are required by certain supernova (SN) explodability prescriptions. In addition to the initial and final conditions, the oneline DataFrame also includes derived quantities such as SN_type.

For a detailed description of the oneline DataFrame and its contents, see the POSYDON documentation:
https://posydon.org/POSYDON/latest/tutorials-examples/population-synthesis/10_binaries_pop_syn.html#Population.oneline

</span>


<div class='alert alert-warning'>


**Important**: When creating your own population runs, ensure that the output columns required in the .ini file are properly specified. Many columns in both the history DataFrame and oneline DataFrame are not saved by default. For the history DataFrame, set the only_select_columns to include the desired attributes of the binary, star_1, and star_2 objects. For the oneline DataFrame, make sure the relevant scalar_names for these objects are included.

</div>

In [ ]:
#convert it to a dataframe
df_oneline=pop.oneline.select()

<span style="font-size:15px">

Now that you are familiar with the `oneline` and `history` DataFrames and know how to access the information they contain, you can explore how to distinguish different supernova explosion mechanisms, such as electron-capture supernovae (ECSNe) and core-collapse supernovae (CCSNe).

The `SN_type` column, stored in the `oneline` DataFrame, provides information about the supernova mechanism associated with each event. Depending on the supernova prescription used in the simulation, this column can identify different outcomes, including ECSNe, pair-instability supernovae (PISNe), pulsational pair-instability events (PPIs), and iron core-collapse supernovae.

</span>


<div class='alert alert-warning'>

**Note: This is the mechanism of explosion, NOT the observational type**


##### Combining `oneline` SN types with `history` stellar evolution properties

In [ ]:
df1_oneline = df_oneline["S1_SN_type"]
df1_oneline.rename('SN_type', inplace=True)

df2_oneline= df_oneline["S2_SN_type"]
df2_oneline.rename('SN_type', inplace=True)

df1_merged = df1.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


df_synthetic_plus_oneline=pd.concat([df1_merged, df2_merged], axis=0)


We now explore the outcome of the SN population, including the supernova type(`SN_type`) information obtained from the `oneline` dataset.

In [ ]:
df_synthetic_plus_oneline.head(10)

<span style="font-size:15px">

Now that we have combined the information from both the history and oneline dataframes for the population file, we can identify the systems—both primary and secondary—that produce ECSNe.

</span>


In [ ]:
ecsn_pop = df_synthetic_plus_oneline[(df_synthetic_plus_oneline["SN_type"] == "ECSN")]
ecsn_pop.tail(10)

<div class="alert alert-success">
<span style="font-size:15px">

## 🛠️ Hands-on Exercise: 
Calculate the fraction of neutron stars (NSs) formed through ECSNe and CCSNe relative to the total NS population, and visualize the resulting fractions in a pie chart. Replace the XX placeholders in the code below with the appropriate expressions.

In [ ]:
# Count CCSNe
ALL_CCSN = (
    ( XX? ).sum() 
)

# Count ECSNe
ALL_ECSN = (
    ( XX? ).sum()
)

print("The number of all CCSNe is", ALL_CCSN)
print("The number of all ECSNe is", ALL_ECSN)



<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<details>

<b><summary>Solution </summary></b>
```

ALL_ECSN = ((df_synthetic_plus_oneline["stellar_state_postCC"].isin(["NS"])) & (df_synthetic_plus_oneline["SN_type"] == "ECSN")).sum()
ALL_CCSN = ((df_synthetic_plus_oneline["stellar_state_postCC"].isin(["NS"])) & (df_synthetic_plus_oneline["SN_type"] == "CCSN")).sum()




```
    
</details>

After estimating the number of neutron stars (NSs) formed through CCSNe and ECSNe, visualize their relative contributions using the following pie-chart code:

In [ ]:
labels = ['CCSN', 'ECSN']
sizes = [ALL_CCSN, ALL_ECSN]
colors = ['grey', 'green']



plt.figure(figsize=(2.5, 2.5))
wedges, texts, autotexts = plt.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}
)


for autotext in autotexts:
    autotext.set_fontsize(14)
    autotext.set_color("black")

plt.axis('equal')  # Keep it circular
plt.title("Collapse mechanism", fontsize=8)
plt.tight_layout()
plt.show()

# Observational SN types

<img src="./SN_taxonomy.png" style="display:block;margin:auto;width:90%">


## Classification of Observed SN Types

<span style="font-size:15px">

To classify pre-CC models as progenitors of **H-poor** (Ib, Ic, IIb)  and **H-rich** (II) supernovae,
we use the total H mass in the ejecta (𝑀H,ej) as a primary indicator,
along with the pre-SN surface He4 and N14 abundances evaluated at the time of core carbon depletion. However, the relationship between the structure of the pre-SN progenitor and the spectroscopic characteristics of the explosion remains an active area of research (e.g., Dessart et al. 2018), we will just use some criteria that are suggested in the literature:
</span>


#### Assumed Criteria based on literature 
(Gilkis et al. 2019, Yoon et al. 2017; Sravan et al. 2018, Aguilera-Dena et al. 2023, Dessart et al. 2020)
- **Ic:** $~M_{H,ej}$ < 0.033 $M_{\odot}$ $~\&~$  $X_{N,surf}$ < 1e-4 & $~\&~$  $X_{He4, surf}$ < 0.5
- **Ib:** $~M_{H,ej}$ < 0.033 $M_{\odot}$ $~\&~$  ($X_{N,surf}$ >= 1e-4 & $~OR~$  $X_{He4, surf}$ >= 0.5)
- **IIb:**  0.033 $M_{\odot}$ <= $M_{H,ej}$ <= 0.5 $M_{\odot}$
- **II:** $~M_{H,ej}$ > 0.5 $M_{\odot}$

#### Color Convention
- <span style="color:cyan; font-weight:bold">Type Ic</span> — cyan 
- <span style="color:blue; font-weight:bold">Type Ib</span> — blue  
- <span style="color:gold; font-weight:bold">Type IIb</span> — yellow  
- <span style="color:red; font-weight:bold">Type II</span> — red  


<span style="font-size:15px">


Below is the function we developed to classify each supernova (SN) event as Type Ic, Ib, IIb, or II. Please review the function carefully to understand the parameters and dataframe columns used for the classification.

In addition, we now need to retain the surface nitrogen and helium abundances at the time of core-carbon depletion, as well as the hydrogen mass ejected during the SN event. The population dataframe should therefore be modified to include these additional quantities for each system.

</span>


In [ ]:
def classify_observed_SN(
    df,
    M_H_Ib=0.033,      # [Msun] Hydrogen ejecta threshold between Ib/Ic and IIb (Gilkis+2019)
    M_H_II=0.5,        # [Msun] Hydrogen ejecta threshold between IIb and II (Yoon+2017; Sravan+2018)
    N_surf_Ic=1e-4,    # [-] Surface nitrogen threshold to distinguish Ic from Ib (Aguilera-Dena+2023)
    he4_surf_Ic= 0.5,   # [-] Surface nitrogen threshold to distinguish Ic from Ib (Aguilera-Dena+2023)
    m_col="h1_mass_ej",  # Column containing hydrogen ejecta mass
    n_col="surface_n14",  # Column containing surface nitrogen abundance
    he4_col="surface_he4",  # Column containing surface helium abundance
    state="stellar_state_postCC"  # Column containing the state of the star after CC1 or CC2

):
    """
    

    The classification is based on the hydrogen ejecta mass (M) and surface 
    nitrogen abundance (N), surface Helium abundance (He4) using thresholds from the literature.

    Rules:
      - Type Ic :  M < M_H_Ib  and  N <  N_surf_Ic and h < he4_surf_Ic
      - Type Ib :  M < M_H_Ib  and  (N >= N_surf_Ic OR h >= he4_surf_Ic)
      - Type IIb:  M_H_Ib < M < M_H_II
      - Type II :  M >= M_H_II
      - Otherwise: "Unknown"

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe, one row per progenitor.
    M_H_Ib : float
        Threshold hydrogen ejecta mass (Msun) separating Ib/Ic from IIb.
    M_H_II : float
        Threshold hydrogen ejecta mass (Msun) separating IIb from II.
    N_surf_Ic : float
        Threshold surface nitrogen abundance distinguishing Ic from Ib.
    m_col : str
        Name of the column in df containing hydrogen ejecta mass.
    n_col : str
        Name of the column in df containing surface nitrogen abundance.

    Returns
    -------
    df : pandas.DataFrame
        Dataframe with one new column:
          - "SN_observed": the SN subtype (II, IIb, Ib, Ic, or Unknown).
    """

    # --- Check that required columns exist ---
    if m_col not in df.columns or n_col not in df.columns:
        raise KeyError(f"Expected columns '{m_col}' and '{n_col}' in df.")

    # Extract relevant quantities
    M = df[m_col]   # hydrogen ejecta mass
    N = df[n_col]   # surface nitrogen abundance
    h = df[he4_col] # surface helium abundance
    s= df[state] # post-CC state

    # Start with everything labeled as Unknown
    observed_type = df.index.to_series().map(lambda _: "Unknown")

    # Apply classification rules
    is_Ic  = (M < M_H_Ib) & (N <  N_surf_Ic) & (h <  he4_surf_Ic) & (s=="NS")
    is_Ib  = (M < M_H_Ib) & ((N >= N_surf_Ic) | (h >=  he4_surf_Ic)) & (s=="NS")
    is_IIb = (M >= M_H_Ib) & (M <  M_H_II) & (s=="NS")
    is_II  = (M >= M_H_II) & (s=="NS")

    observed_type = observed_type.mask(is_Ic,  "Ic")
    observed_type = observed_type.mask(is_Ib,  "Ib")
    observed_type = observed_type.mask(is_IIb, "IIb")
    observed_type = observed_type.mask(is_II,  "II")

    # Save results back into the dataframe
    df["SN_observed"] = observed_type

    return df


<div class="alert alert-success">
<span style="font-size:15px">

## 🛠️ Hands-on Exercise: Find the relative rates of all SN Types in the population

1. Based on the assumed classification criteria above and the inputs used in the `classify_observed_SN`, determine which properties and columns need to be retained.

2. Replace the XXX values in the code below and create the new population keeping the information that should be used in the `classify_observed_SN` function.
   
</div>

<details>
<summary><b>Hint 1</b></summary>
Some properties/columns describe the preCC (progenitor) state, others pertain to the compact object, and a few are listed directly in the oneline dataframe since they were pre-calculated from the profile.
</details>

<details>
<summary><b>Hint2</b></summary>
We should modify the script below to include information of the surface abundances of the stars 
before SNe (at core carbon depletion) adding the appropriate columns.

- **preSN** line → progenitor properties (e.g., surface abundances, core mass etc). In our case we need `surface_he4`,`surface_n14`, preSN mass of the progenitor
- **postSN** line → compact object mass and state, orbit re-adjustment after instataneous mass loss and natal kick, etc. In our case we need compact object mass and state
- **oneline** dataframe → keeping some usually pre_computed info about the event. In our case we need `h1_mass_ej`

 </details>

In [ ]:
# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.
# -----------------------------------------------------------------------------
pre_CC1 = (df['event'] == XXX )
pre_CC2 = (df['event'] == "CC2")


# -----------------------------------------------------------------------------
# Extract the surface composition of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.
# -----------------------------------------------------------------------------
df1_pre = df[['S1_surface_n14', XXX]][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.
# -----------------------------------------------------------------------------
df2_pre = df[['S2_surface_he4', 'S2_surface_n14']][XXX].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4'
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.
# -----------------------------------------------------------------------------

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', XXX]].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.
# -----------------------------------------------------------------------------
df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties
# -----------------------------------------------------------------------------
df_synthetic_plus_oneline_plus_pre_CC = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

<div class="alert alert-danger alertdanger" style="margin-top: 20px">
<details>

<b><summary>Solution  </summary></b>
```

# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.
# -----------------------------------------------------------------------------
pre_CC1 = (df['event'] == "CC1")
pre_CC2 = (df['event'] == "CC2")


# -----------------------------------------------------------------------------
# Extract the surface composition of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.
# -----------------------------------------------------------------------------
df1_pre = df[['S1_surface_n14', 'S1_surface_he4']][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.
# -----------------------------------------------------------------------------
df2_pre = df[['S2_surface_he4', 'S2_surface_n14']][pre_CC2].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4'
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.
# -----------------------------------------------------------------------------

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', 'S1_h1_mass_ej']].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.
# -----------------------------------------------------------------------------
df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties
# -----------------------------------------------------------------------------
df_synthetic_plus_oneline_plus_pre_CC = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

```
    
</details>

In [ ]:
df_synthetic_plus_oneline_plus_pre_CC.head(20)

<span style="font-size:15px">


Having extracted the relevant stellar properties required by the SN classification function, we can now estimate the relative rates of the different observational SN types (Ic, Ib, IIb, and II) within the population.

These relative rates are calculated from the number of systems classified as each SN type, normalized to the total number of classified SN events. Absolute SN rates, however, would require an additional normalization by the total mass formed in the stellar population.

</span> 


In [ ]:
import matplotlib.pyplot as plt

# Classify observed SN types with your chosen thresholds
df_classified = classify_observed_SN(df_synthetic_plus_oneline_plus_pre_CC, M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4)

# Collect observed SN types for both stars
obs = df_classified["SN_observed"]
obs = obs[obs.isin(["Ic", "Ib", "IIb", "II"])]  # drop Unknown if present

# Count and normalize
counts = obs.value_counts().reindex(["Ic", "Ib", "IIb", "II"], fill_value=0)
fractions = counts.values.astype(float)
fractions = fractions / fractions.sum()

# Labels and colors
labels = ["Type Ic", "Type Ib", "Type IIb", "Type II"]
colors = ["cyan", "blue", "yellow", "red"]
explode = [0.05] * 4

# Plot pie chart
plt.figure(figsize=(2.0, 2.0))
wedges, texts, autotexts = plt.pie(
    fractions,
    labels=labels,
    colors=colors,
    explode=explode,
    autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}
)

for autotext in autotexts:
    autotext.set_fontsize(7)
    autotext.set_color("black")

plt.title("Fraction of Observed Supernova Types")
plt.show()


##  Ejecta masses of different types of SNe

Another especially important parameter in SN studies, is the ejecta mass, Mej, because it can be used to discriminate between different origins of different types of SNe, and relating it to the zero-age main-sequence (ZAMS) mass, teaches us about the mass loss process. Below, we demonstrate how the pre-SN mass and remnant mass can be used to determine the ejecta masses of Type Ib core-collapse supernovae. We restrict the analysis to events that form neutron stars NSs, excluding systems that form BHs, for which additional complications related to fallback may arise.

In [ ]:
# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.
# -----------------------------------------------------------------------------
pre_CC1 = (df['event'] == "CC1") 
pre_CC2 = (df['event'] == "CC2") 


# -----------------------------------------------------------------------------
# Extract the surface composition and the mass of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.
# -----------------------------------------------------------------------------
df1_pre = df[['S1_surface_n14', 'S1_surface_he4', 'S1_mass']][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4',
    'S1_mass': 'preCC_mass'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.
# -----------------------------------------------------------------------------
df2_pre = df[['S2_surface_he4', 'S2_surface_n14', 'S2_mass']][pre_CC2].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4',
    'S2_mass': 'preCC_mass'

    
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.
# -----------------------------------------------------------------------------

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', 'S1_h1_mass_ej']].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.
# -----------------------------------------------------------------------------
df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties
# -----------------------------------------------------------------------------
df_synthetic_plus_oneline_plus_pre_CC_ejecta = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

    



In [ ]:
df_synthetic_plus_oneline_plus_pre_CC_ejecta.head(10)

Having estimated the pre-core-collapse mass and compact-object mass, we can derive the ejecta masses. By combining these results with the classify_observed_SN function to identify the observational SN types, we can construct histograms of the ejecta-mass distribution for Type Ib SNe. The distribution is considered separately for Type Ib SNe originating from the primary, secondary and single stars.

In [ ]:
# Run classification on the underlying DataFrame
df_classified = classify_observed_SN(
    df_synthetic_plus_oneline_plus_pre_CC_ejecta,
    M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4
)

# Compute ejecta mass = preSN mass - remnant mass  
df_classified["M_ejecta"] = df_classified["preCC_mass"] - df_classified["compact_object_mass"]

is_NS = df_classified['stellar_state_postCC'].eq('NS')

# Observed Type Ib masks
is_Ib = df_classified['SN_observed'].eq('Ib')

M_ej_Ib_NS = pd.concat([
    df_classified.loc[is_NS & is_Ib, 'M_ejecta'],
]).dropna()
M_ej_Ib_NS = M_ej_Ib_NS[M_ej_Ib_NS > 0]

# Plot
plt.figure(figsize=(3,3))
plt.hist(M_ej_Ib_NS, bins=15, alpha=0.85)
plt.xlabel(r"$M_{\mathrm{ej}}$ [M$_\odot$]")
plt.ylabel("Count")
plt.title("Ejecta masses: Type Ib CCSNe with NS remnants")
plt.tight_layout()
plt.show()


### Formation pathways of SNe

While you can see all the main evolutionary steps in the evolution of a binary, it is useful to have a summary overview of a binary’s evolutionary pathway, also known as a formation channel.

You might be interested in figuring out what sort of formation pathways/channels a binary has followed throughout its evolution.



In [ ]:
pop.calculate_formation_channels(mt_history=True)

In [ ]:
df_channels=pop.formation_channels

In [ ]:
df_synthetic_with_channels = df_synthetic_plus_oneline_plus_pre_CC_ejecta.merge(
    df_channels,
    left_index=True,
    right_index=True,
    how="inner"
)

In [ ]:
df_synthetic_with_channels.head(10)

The formation channels are calculated by combining the event column in the history table into a single string using the calculate_formation_channels() function of the Population object.

Two columns are available in the formation channels table:

    debug_channel : A longer description of the formation channel, where additional events are included.

    channel : A cleaned-up version of the history events, where events are separated by a -.


### Binary Evolution Channel

The `channel` column provides a condensed summary of the binary evolution by retaining only the key evolutionary events.

All evolutionary sequences begin with **ZAMS**, representing the zero-age main sequence. The subsequent entries describe the major interactions and evolutionary stages of the system.

* **`oRLO1` / `oRLO2`** — onset of Roche-lobe overflow (RLO), indicating the beginning of mass transfer from star 1 or star 2, respectively.
* **`oCE1` / `oCE2`** — onset of a common-envelope (CE) phase following unstable mass transfer from star 1 or star 2.
* **`oDoubleCE1` / `oDoubleCE2`** — onset of a double common-envelope phase.
* **`CC1` / `CC2`** — core collapse of star 1 or star 2, marking the end of that star's evolution and the formation of a compact remnant.
* **`merging1`** — merger of the binary during a common-envelope phase initiated by star 1. If an `oCE1` event is not followed by `merging1`, the system is considered to have survived the common-envelope phase.
* **`merging2`** — analogous merger following a common-envelope phase initiated by star 2.

If an `oRLO1` or `oRLO2` event is **not followed by the corresponding `oCE` event**, the mass transfer is assumed to be **stable**. Conversely, if an RLO event is followed by `oCE`, the mass transfer is classified as **unstable**, leading to a common-envelope phase.

Systems that contain no `oRLO`, `oCE`, or `oDoubleCE` events are classified as **non-interacting binaries**, meaning that the stars evolve without undergoing significant binary interaction.

Thus, the `channel` column provides a compact representation of the evolutionary pathway, allowing the different binary-interaction channels leading to compact-object formation to be identified and compared.


Let's identify the evolutionary channels of star 1 and star 2 that lead to the production of Type Ib supernovae, and determine the most dominant evolutionary pathway.

In [ ]:
df_classified = classify_observed_SN(
    df_synthetic_with_channels,
    M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4
)

In [ ]:
df_classified.head(10)

In [ ]:
typeIb_SNe_from_star1 = ((df_classified['stellar_state_postCC']== 'NS') & 
        (df_classified['SN_observed']== 'Ib') &
        (df_classified['progenitor_star']==1) &
        (df_classified['binary_state_postCC']!='initially_single_star'))

df_classified['channel_debug'][typeIb_SNe_from_star1].value_counts()


In [ ]:
typeIb_SNe_from_star1 = ((df_classified['stellar_state_postCC']== 'NS') & 
        (df_classified['SN_observed']== 'Ib') &
        (df_classified['progenitor_star']==2) &
        (df_classified['binary_state_postCC']!='initially_single_star'))

df_classified['channel_debug'][typeIb_SNe_from_star1].value_counts()


Having completed the analysis above, we now have an idea of the dominant evolutionary channels leading to Type Ib supernovae from both primary and secondary stars. However, the `channel` information does not provide the detailed mass-transfer history. For example, when an `oRLO1` event is not followed by a common-envelope event, we can identify the mass transfer as stable, but we cannot determine the specific mass-transfer case (Case A, B, or C) or the number of distinct mass-transfer episodes that occurred.

To obtain a more detailed picture of the evolutionary pathways, we can perform a parameter-space survey by overlaying our Type Ib progenitor systems onto the POSYDON evolutionary grids for different initial mass ratios (q). We can examine the systems as a function of **initial primary mass** and **initial orbital period**, and color-code the grid according to the corresponding mass-transfer history. This will allow us to identify where Type Ib progenitors are located in parameter space and determine which mass-transfer cases and evolutionary pathways dominate their formation.


Adding the initial binary conditions to the df_synthetic_plus_oneline_plus_pre_CC_ejecta dataframe

In [ ]:
# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.
# -----------------------------------------------------------------------------
pre_CC1 = (df['event'] == "CC1") 
pre_CC2 = (df['event'] == "CC2") 

step_ini = (df['step_names'] == "initial_cond") #initial conditions


# -----------------------------------------------------------------------------
# Extract the surface composition and the mass of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.
# -----------------------------------------------------------------------------
df1_ini = df[['S1_mass','S2_mass','orbital_period']][step_ini].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_ini = df1_ini.rename(columns={
    'S1_mass': 'S1_mass_ini',
    'S2_mass': 'S2_mass_ini',
    'orbital_period': 'period_ini'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_k = pd.merge(
    df1,
    df1_ini,
    left_index=True,
    right_index=True,
    how='inner'
)



# -----------------------------------------------------------------------------
# Extract the surface composition and the mass of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.
# -----------------------------------------------------------------------------
df1_pre = df[['S1_surface_n14', 'S1_surface_he4', 'S1_mass']][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4',
    'S1_mass': 'preCC_mass'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_n = pd.merge(
    df1_k,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.
# -----------------------------------------------------------------------------
df2_pre = df[['S2_surface_he4', 'S2_surface_n14', 'S2_mass']][pre_CC2].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4',
    'S2_mass': 'preCC_mass'

    
})

df2_k = pd.merge(
    df2,
    df1_ini,
    left_index=True,
    right_index=True,
    how='inner'
)


df2_n = pd.merge(
    df2_k,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.
# -----------------------------------------------------------------------------

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', 'S1_h1_mass_ej']].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.
# -----------------------------------------------------------------------------
df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties
# -----------------------------------------------------------------------------
df_synthetic_plus_oneline_plus_pre_CC_ejecta_ini = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

    



In [ ]:
df_synthetic_plus_oneline_plus_pre_CC_ejecta_ini.tail(10)

In [ ]:
#Since the initial mass ratio (q) is not directly provided as a column in the dataframe, we calculate it from the initial masses of the two stars:
df_synthetic_plus_oneline_plus_pre_CC_ejecta_ini['q']=df_synthetic_plus_oneline_plus_pre_CC_ejecta_ini['S2_mass_ini']/df_synthetic_plus_oneline_plus_pre_CC_ejecta_ini['S1_mass_ini']

df_classified = classify_observed_SN(
    df_synthetic_plus_oneline_plus_pre_CC_ejecta_ini,
    M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4
)



# Select Type Ib SNe from systems with q ≈ 0.4


mask = (df_classified['SN_observed']=='Ib') & (df_classified['q']>0.4-0.015) & (df_classified['q']<0.4+0.015) 


#### Loading the HMS-HMS grid as you learned in the exploring grids lab.

In [ ]:
import os
from posydon.config import PATH_TO_POSYDON_DATA
from posydon.grids.psygrid import PSyGrid
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Shared plot style — keeps every figure in this notebook visually consistent.
# (Purely cosmetic: does not change any plotting logic below.)
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.axisbelow': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'legend.frameon': False,
    'lines.linewidth': 1.8,
    'axes.prop_cycle': plt.cycler(color=["#77538F", "#589393", "#A4AA92", "#403B31", "#804727"]),
})

In [ ]:
grid_file_path = os.path.join(PATH_TO_POSYDON_DATA, 'HMS-HMS', '1e+00_Zsun.h5') 

In [ ]:
grid = PSyGrid()
grid.load(grid_file_path)

In [ ]:
from posydon.visualization.plot2D import plot2D

PLOT_PROPERTIES = {
    'show_fig' : True,
    'close_fig' : True,
    'figsize' : (6,6),
    'legend1D': dict(loc='upper right', lines_legend=['9000','1000']),
    'log10_x' : True,
    'log10_y' : True,
}

fig, ax = plt.subplots(figsize=(3.5, 3.1))
q=0.4
pp = plot2D(
    grid,
    'star_1_mass',
    'period_days',
    None,
    termination_flag='termination_flag_2',
    grid_3D=True,
    slice_3D_var_str='mass_ratio',
    slice_3D_var_range=(q - 0.015, q + 0.015),
    verbose=False,
    **PLOT_PROPERTIES
)

pp.plot_panel(ax)

ax.scatter(
    np.log10(df_classified['S1_mass_ini'][mask]),
    np.log10(df_classified['period_ini'][mask])
)

plt.show()

<div class="alert alert-success">
<span style="font-size:15px">

## 🛠️ Hands-on Exercise: 
If you finish early and have time, repeat the analysis for different SN types and initial mass ratios. Identify the dominant evolutionary channels producing Type IIb and Type II SNe, and then overplot the corresponding results on the HMS–HMS grid for different values of q.
   
</div>

## Van den Heuvel diagrams with POSYDON (optional)

If you install POSYDON on your computer following the instructions provided in the linked guide, you’ll have the option to enable experimental visualization libraries. While these libraries offer advanced features, please note that they might still be in development and could be subject to changes.
Instructions for visualizations: https://posydon.org/POSYDON/latest/getting-started/installation-guide.html#id11

To install these experimental visualization libraries
Navigate to your POSYDON directory (where the setup.py is located) and run:
pip install ".[vis]"

Unfortunataly, this will not work in the environemnt for the School. 
See the example commands below. TAs will show an example diagram.

from posydon.visualization.VHdiagram import VHdiagram
VHdiagram('SNe_lab1.h5', path='./', index=4)